In [1]:
import pandas as pd
import numpy as np
import pickle as pkl
from geolib import geohash as geolib
from numpy.random import default_rng
from statsmodels.stats.multitest import multipletests

In [2]:
paths = pd.read_csv("data/all_paths_128_cleaned.csv", sep=";")
movements = pd.read_csv("data/all_movements_117.csv", sep=";")

In [3]:
# Read geohashes to coordinates mapping (dataframe with columns: geohash, latitude, longitude)
geo_to_coords = pd.read_pickle('geohashes_to_coords.pkl')

# Function to complete geo_to_coords with missing geohashes
def complete_geo_to_coords(geo_to_coords, geohashes):
    missing_geohashes = set(geohashes) - set(geo_to_coords['geohash'])
    new_rows = []
    for gh in missing_geohashes:
        lat, lon = geolib.decode(gh)
        new_rows.append({'geohash': gh, 'latitude': lat, 'longitude': lon})
    if new_rows:
        geo_to_coords = pd.concat([geo_to_coords, pd.DataFrame(new_rows)], ignore_index=True)
        geo_to_coords.to_pickle('geohashes_to_coords.pkl') # Save to file
    return geo_to_coords

In [4]:
swiss_geohashes = [f"u0{c}" for c in "kmqrhjnp"]
paths["geohash"] = paths["geohash"].str.slice(0, 8)
paths = paths[paths["geohash"].str.startswith(tuple(swiss_geohashes))]

In [5]:
GEO_LEN = 7   # recommended for CH-wide
paths["cell"] = paths["geohash"].str.slice(0, GEO_LEN)

In [6]:
all_geohashes = set(paths['geohash'].dropna().unique())
geo_to_coords = complete_geo_to_coords(geo_to_coords, all_geohashes)

In [7]:
paths["date"] = pd.to_datetime(paths["date"], errors="coerce").dt.date

heatwave_days = pd.to_datetime([
    "2025-06-13", "2025-06-14", "2025-06-22", "2025-06-25",
    "2025-06-29", "2025-06-30",
    "2025-07-01", "2025-07-02",
    "2025-08-09", "2025-08-12", "2025-08-13",
    "2025-08-14", "2025-08-15"
]).date

In [8]:
def translate_mot(mot):
    if mot in ["CAR", "ELECTRIC_CAR", "HYBRID_CAR"]:
        return "Car"
    elif mot == "TRAIN":
        return "Train"
    elif mot == "WALKING":
        return "Walking"
    elif mot in ["ON_BICYCLE", "ELECTRIC_BIKE", "SCOOTER", "ELECTRIC_SCOOTER"]:
        return "Bicycle"
    elif mot in ["BUS", "ELECTRIC_BUS", "COACH"]:
        return "Bus"
    elif mot == "TRAM":
        return "Tram"
    elif mot == "PLANE":
        return "Plane"
    elif mot in ["BOAT", "BOAT_NO_ENGINE"]:
        return "Boat"
    else:
        return mot

paths["mean_of_transport"] = paths["mode_of_transport"].apply(translate_mot)


In [9]:
paths["date_dt"] = pd.to_datetime(paths["date"])
paths["weekday"] = paths["date_dt"].dt.weekday
paths["month"] = paths["date_dt"].dt.month
paths["is_heatwave_day"] = paths["date"].isin(heatwave_days)

cell_day = (
    paths
    .groupby(["cell", "date", "month", "weekday", "mean_of_transport", "is_heatwave_day"])
    .size()
    .reset_index(name="count")
)


In [10]:
movements["start_time"] = pd.to_datetime(movements["start_time"])
movements["date"] = movements["start_time"].dt.date

active_users = (
    movements.groupby("date")["participant_id"]
    .nunique()
    .reset_index(name="n_users")
)

df = cell_day.merge(active_users, on="date", how="left")
df["norm_intensity"] = df["count"] / df["n_users"]


In [11]:
baseline = (
    df[df["is_heatwave_day"] == 0]
    .groupby(["cell", "mean_of_transport"])["norm_intensity"]
    .median()
    .rename("cell_baseline")
    .reset_index()
)

df = df.merge(baseline, on=["cell", "mean_of_transport"], how="left")
df["norm_cell_intensity"] = df["norm_intensity"] / df["cell_baseline"]

In [12]:
modes_of_interest = ["Walking", "Bicycle", "Boat"]
df = df[df["mean_of_transport"].isin(modes_of_interest)]

In [13]:
non_hw = df[df["is_heatwave_day"] == 0][["date", "month", "weekday"]].drop_duplicates()
hw_days = df[df["is_heatwave_day"] == 1][["date", "month", "weekday"]].drop_duplicates()

controls_by_hw = {}
for _, r in hw_days.iterrows():
    ctrls = non_hw[
        (non_hw["month"] == r["month"]) &
        (non_hw["weekday"] == r["weekday"])
    ]["date"].tolist()
    controls_by_hw[r["date"]] = ctrls

In [14]:
VALUE_COL = "norm_cell_intensity"
pivot = df.pivot_table(index="cell", columns="date", values=VALUE_COL, aggfunc="mean")

effects = []
for cell in pivot.index:
    deltas = []
    for hd, ctrls in controls_by_hw.items():
        if hd not in pivot.columns:
            continue
        hw_val = pivot.at[cell, hd]
        if pd.isna(hw_val):
            continue
        ctrl_vals = pivot.loc[cell].reindex(ctrls).dropna()
        if len(ctrl_vals) == 0:
            continue
        deltas.append(hw_val - ctrl_vals.median())
    if deltas:
        effects.append((cell, np.median(deltas), len(deltas)))

effects_df = pd.DataFrame(effects, columns=["cell", "delta_hw_vs_ctrl", "n_matched"])


In [15]:
rng = default_rng(0)

def signflip_pvalue(x, n=5000):
    obs = np.median(x)
    sims = np.median(x * rng.choice([-1, 1], size=(n, len(x))), axis=1)
    return (np.sum(np.abs(sims) >= abs(obs)) + 1) / (n + 1)

pvals = []
for cell in pivot.index:
    deltas = []
    for hd, ctrls in controls_by_hw.items():
        if hd not in pivot.columns:
            continue
        hw_val = pivot.at[cell, hd]
        if pd.isna(hw_val):
            continue
        ctrl_vals = pivot.loc[cell].reindex(ctrls).dropna()
        if len(ctrl_vals) == 0:
            continue
        deltas.append(hw_val - ctrl_vals.median())
    if len(deltas) >= 1:
        pvals.append((cell, signflip_pvalue(np.array(deltas))))
    else:
        pvals.append((cell, np.nan))

p_df = pd.DataFrame(pvals, columns=["cell", "p"])


In [16]:
mask = p_df["p"].notna()
reject, p_adj, _, _ = multipletests(p_df.loc[mask, "p"], method="fdr_bh")
p_df.loc[mask, "p_adj"] = p_adj
p_df.loc[mask, "significant"] = reject


In [17]:
final_df = (
    effects_df
    .merge(p_df, on="cell", how="left")
    .sort_values("delta_hw_vs_ctrl", ascending=False)
)


In [18]:
final_df.rename(columns={"cell": "geohash"}, inplace=True)

In [19]:
# Add coordinates from geo_to_coords
final_df = final_df.merge(geo_to_coords, on='geohash', how="left")

In [20]:
all_geohashes = set(final_df['geohash'].dropna().unique())
geo_to_coords = complete_geo_to_coords(geo_to_coords, all_geohashes)

In [21]:
final_df

,geohash,delta_hw_vs_ctrl,n_matched,p,p_adj,significant,latitude,longitude
0,u0qj99p,5.151316,1,1.000000,1.0,False,47.3792266845703125,8.5137176513671875
1,u0qnnmb,3.400000,1,1.000000,1.0,False,47.49321,8.712845
2,u0qj9c5,2.920902,1,1.000000,1.0,False,47.379227,8.519211
3,u0qj99r,2.471339,2,0.500700,1.0,False,47.3806,8.513718
4,u0qtp31,2.375000,1,1.000000,1.0,False,47.2913360595703125,9.4612884521484375
...,...,...,...,...,...,...,...,...
586,u0m7149,-3.811765,1,1.000000,1.0,False,46.948013,7.428818
587,u0m70f6,-4.500000,1,1.000000,1.0,False,46.94664,7.419205
588,u0m709v,-4.807018,3,0.504899,1.0,False,46.943893,7.412338
589,u0m709f,-5.850000,3,0.504899,1.0,False,46.943893,7.408218


In [22]:
# Remove rows with missing coordinates
final_df = final_df.dropna(subset=['latitude', 'longitude'])
final_df

,geohash,delta_hw_vs_ctrl,n_matched,p,p_adj,significant,latitude,longitude
0,u0qj99p,5.151316,1,1.000000,1.0,False,47.3792266845703125,8.5137176513671875
1,u0qnnmb,3.400000,1,1.000000,1.0,False,47.49321,8.712845
2,u0qj9c5,2.920902,1,1.000000,1.0,False,47.379227,8.519211
3,u0qj99r,2.471339,2,0.500700,1.0,False,47.3806,8.513718
4,u0qtp31,2.375000,1,1.000000,1.0,False,47.2913360595703125,9.4612884521484375
...,...,...,...,...,...,...,...,...
586,u0m7149,-3.811765,1,1.000000,1.0,False,46.948013,7.428818
587,u0m70f6,-4.500000,1,1.000000,1.0,False,46.94664,7.419205
588,u0m709v,-4.807018,3,0.504899,1.0,False,46.943893,7.412338
589,u0m709f,-5.850000,3,0.504899,1.0,False,46.943893,7.408218


In [23]:
import folium

m = folium.Map(
    location=[46.8, 8.3],
    zoom_start=8,
    tiles="CartoDB positron"
)

for _, r in final_df.iterrows():
    if pd.isna(r["delta_hw_vs_ctrl"]) or pd.isna(r["latitude"]):
        continue

    color = "red" if r["delta_hw_vs_ctrl"] > 0 else "blue"
    opacity = 0.75 if r.get("significant", False) else 0.25

    folium.CircleMarker(
        location=[r["latitude"], r["longitude"]],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=opacity,
        weight=0
    ).add_to(m)

m.save("maps/heatwave_effects_map.html")

!open maps/heatwave_effects_map.html

In [29]:
strong = final_df[
    #(final_df["significant"]) &
    (final_df["delta_hw_vs_ctrl"].abs() > final_df["delta_hw_vs_ctrl"].quantile(0.9))
]

m2 = folium.Map(location=[46.8, 8.3], zoom_start=8, tiles="CartoDB positron")

for _, r in strong.iterrows():
    folium.CircleMarker(
        location=[r["latitude"], r["longitude"]],
        radius=5,
        color="red" if r["delta_hw_vs_ctrl"] > 0 else "blue",
        fill=True,
        fill_opacity=0.8,
        weight=0
    ).add_to(m2)

m2.save("maps/strong_heatwave_effects_map.html")

!open maps/strong_heatwave_effects_map.html
